In [6]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd

url = "https://data.wprdc.org/datastore/dump/5202679a-d243-402e-b82a-63189995a942?format=csv"
df = pd.read_csv(url,dtype ="string")

In [8]:

pittsburgh = df.loc[
    df["city"].str.strip().str.casefold().eq("pittsburgh")
    & df["neighborhood"].notna()
    & df["neighborhood"].str.strip().ne("")
].copy()

pittsburgh["neighborhood"] = pittsburgh["neighborhood"].str.strip()

roadwork = pittsburgh.loc[
    pittsburgh["subject_code"].str.strip().str.upper().eq("ROADWORK")
].copy()

print(f"Matching ROADWORK records: {len(roadwork):,}")


Matching ROADWORK records: 5,429


In [9]:

neighborhoods = sorted(pittsburgh["neighborhood"].unique())


if roadwork.empty:
    raise ValueError("No ROADWORK records found. Check your filters.")


counts = roadwork.groupby("neighborhood")["unique_id"].nunique()


counts = counts.reindex(neighborhoods, fill_value=0)


results = counts.rename("roadwork_requests").reset_index()

highest_count = results["roadwork_requests"].max()

if highest_count == 0:
    raise ValueError("No valid request IDs found. Check the unique_id column.")

results["score"] = (
    100 * (1 - results["roadwork_requests"] / highest_count)
).round(2)


results["rank"] = (
    results["roadwork_requests"]
    .rank(method="min", ascending=True)
    .astype(int)
)

results = results.sort_values(
    ["roadwork_requests", "neighborhood"]
)

results = results[
    ["rank", "neighborhood", "roadwork_requests", "score"]
]

print(results.to_string(index=False))

 rank              neighborhood  roadwork_requests  score
    1         Arlington Heights                  0 100.00
    1                Glen Hazel                  0 100.00
    1           Southside Flats                  0 100.00
    1          Southside Slopes                  0 100.00
    5             Block Group 4                  1  99.57
    5             Block Group 5                  1  99.57
    5             East Carnegie                  1  99.57
    5                 Fairywood                  1  99.57
    9                 St. Clair                  2  99.14
   10             Block Group 2                  3  98.71
   10             Block Group 3                  3  98.71
   10            Chartiers City                  3  98.71
   13             Block Group 1                  5  97.85
   14                    Esplen                  6  97.42
   14             Homewood West                  6  97.42
   14         Northview Heights                  6  97.42
   14         